# Core Validation 003 — Dependency-Scoped Transactional Continual Learning

Formal Kaggle workflow for the frozen 003 protocol.

The notebook:
1. checks out the frozen implementation branch,
2. installs the package,
3. runs the 003 unit/integration tests,
4. runs a CPU smoke experiment,
5. requires CUDA for the formal three-seed run,
6. generates reports,
7. optionally publishes curated artifacts to the result branch.

Do not change protocol thresholds after inspecting formal results.


In [ ]:
from pathlib import Path
import os, subprocess, sys

ROOT = Path("/kaggle/working/mini-cells")
BRANCH = "codex/core-validation-003-dependency-scoped-transactional-learning"

if not ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/ArcheLabs/mini-cells.git", str(ROOT)], check=True)
subprocess.run(["git", "fetch", "origin", BRANCH], cwd=ROOT, check=True)
subprocess.run(["git", "checkout", "-B", BRANCH, f"origin/{BRANCH}"], cwd=ROOT, check=True)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip())
os.chdir(ROOT)


In [ ]:
!python -m pip install -q -e ".[dev]"


In [ ]:
!pytest -q tests/test_core_validation_003.py


In [ ]:
!rm -rf results/core-validation-003-dependency-scoped-transactional-learning
!python scripts/run_core_validation_003.py --device cpu --smoke
!python scripts/report_core_validation_003.py


In [ ]:
import torch
assert torch.cuda.is_available(), "Formal Core Validation 003 requires CUDA"
print(torch.cuda.get_device_name(0))


In [ ]:
!rm -rf results/core-validation-003-dependency-scoped-transactional-learning
!python scripts/run_core_validation_003.py --device cuda
!python scripts/report_core_validation_003.py


In [ ]:
import json
from pathlib import Path
import pandas as pd

OUT = Path("results/core-validation-003-dependency-scoped-transactional-learning")
decision = json.loads((OUT / "decision.json").read_text())
display(decision)

gates = pd.read_csv(OUT / "gate-summary.csv")
display(gates[[
    "seed", "granularity", "pass",
    "tx_mean_dependency_coverage", "tx_false_safe_rate",
    "tx_acceptance_rate",
    "regression_damage_ratio_vs_local_always",
    "committed_gain_ratio_vs_local_always",
    "tx_maximum_structural_escape_rate",
    "stress_false_safe_rate",
    "stress_maximum_structural_escape_rate",
]])


In [ ]:
from IPython.display import Image, display
for name in [
    "scope-safety-frontier.png",
    "granularity-scope-acceptance.png",
    "cost-per-accepted-update.png",
    "transactional-tradeoff.png",
]:
    display(Image(filename=str(OUT / name)))


In [ ]:
PUBLISH_RESULTS = True

if PUBLISH_RESULTS:
    subprocess.run([sys.executable, "scripts/publish_core_validation_003.py", "--push"], cwd=ROOT, check=True)
